# Классификация: SI > медианы (с учётом логарифма)

Цель: по дескрипторам предсказать, превышает ли индекс селективности SI медиану выборки. С учётом сильной асимметрии распределения SI дополнительно используется логарифм SI (`SI_log`), который делает задачу более стабильной

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

pd.set_option('display.max_columns', 200)

df = pd.read_csv('/Users/yaroslavbaev/Desktop/miphi/data/chem_data_prepared.csv')

df.head()

,Unnamed: 0,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,MaxPartialCharge,MinPartialCharge,MaxAbsPartialCharge,MinAbsPartialCharge,FpDensityMorgan1,FpDensityMorgan2,FpDensityMorgan3,BCUT2D_MWHI,BCUT2D_MWLOW,BCUT2D_CHGHI,BCUT2D_CHGLO,BCUT2D_LOGPHI,BCUT2D_LOGPLOW,BCUT2D_MRHI,BCUT2D_MRLOW,AvgIpc,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3n,Chi3v,Chi4n,Chi4v,HallKierAlpha,Ipc,Kappa1,Kappa2,Kappa3,LabuteASA,PEOE_VSA1,PEOE_VSA10,PEOE_VSA11,PEOE_VSA12,PEOE_VSA13,PEOE_VSA14,PEOE_VSA2,PEOE_VSA3,PEOE_VSA4,PEOE_VSA5,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,PEOE_VSA9,SMR_VSA1,SMR_VSA10,SMR_VSA2,SMR_VSA3,SMR_VSA4,SMR_VSA5,SMR_VSA6,SMR_VSA7,SMR_VSA8,SMR_VSA9,SlogP_VSA1,SlogP_VSA10,SlogP_VSA11,SlogP_VSA12,SlogP_VSA2,SlogP_VSA3,SlogP_VSA4,SlogP_VSA5,SlogP_VSA6,SlogP_VSA7,SlogP_VSA8,SlogP_VSA9,TPSA,EState_VSA1,EState_VSA10,EState_VSA11,EState_VSA2,EState_VSA3,EState_VSA4,EState_VSA5,EState_VSA6,EState_VSA7,EState_VSA8,EState_VSA9,VSA_EState1,...,NumAliphaticHeterocycles,NumAliphaticRings,NumAromaticCarbocycles,NumAromaticHeterocycles,NumAromaticRings,NumHAcceptors,NumHDonors,NumHeteroatoms,NumRotatableBonds,NumSaturatedCarbocycles,NumSaturatedHeterocycles,NumSaturatedRings,RingCount,MolLogP,MolMR,fr_Al_COO,fr_Al_OH,fr_Al_OH_noTert,fr_ArN,fr_Ar_COO,fr_Ar_N,fr_Ar_NH,fr_Ar_OH,fr_COO,fr_COO2,fr_C_O,fr_C_O_noCOO,fr_C_S,fr_HOCCN,fr_Imine,fr_NH0,fr_NH1,fr_NH2,fr_N_O,fr_Ndealkylation1,fr_Ndealkylation2,fr_Nhpyrrole,fr_SH,fr_aldehyde,fr_alkyl_carbamate,fr_alkyl_halide,fr_allylic_oxid,fr_amide,fr_amidine,fr_aniline,fr_aryl_methyl,fr_azide,fr_azo,fr_barbitur,fr_benzene,fr_benzodiazepine,fr_bicyclic,fr_diazo,fr_dihydropyridine,fr_epoxide,fr_ester,fr_ether,fr_furan,fr_guanido,fr_halogen,fr_hdrzine,fr_hdrzone,fr_imidazole,fr_imide,fr_isocyan,fr_isothiocyan,fr_ketone,fr_ketone_Topliss,fr_lactam,fr_lactone,fr_methoxy,fr_morpholine,fr_nitrile,fr_nitro,fr_nitro_arom,fr_nitro_arom_nonortho,fr_nitroso,fr_oxazole,fr_oxime,fr_para_hydroxylation,fr_phenol,fr_phenol_noOrthoHbond,fr_phos_acid,fr_phos_ester,fr_piperdine,fr_piperzine,fr_priamide,fr_prisulfonamd,fr_pyridine,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0.0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,384.652,340.300,384.350449,158.0,0.0,0.038844,-0.293526,0.293526,0.038844,0.642857,1.035714,1.321429,14.822266,9.700470,2.600532,-2.343082,2.644698,-2.322229,5.944519,0.193481,3.150503,1.164038,611.920301,20.208896,19.534409,19.534409,13.127794,12.204226,12.204226,12.058078,12.058078,10.695991,10.695991,7.340247,7.340247,-0.66,2.187750e+06,20.606247,6.947534,2.868737,173.630124,0.000000,0.0,0.0,0.0,0.0,0.0,9.984809,0.0,0.0,0.0,54.384066,74.032366,35.342864,0.000000,0.000000,11.423370,0.0,0.000000,43.480583,105.750639,13.089513,0.00000,0.0,0.0,0.000000,0.000000,0.0,0.0,24.512883,0.000000,33.495774,105.750639,9.984809,0.0,0.0,0.0,24.72,0.0,0.0,0.0,0.000000,21.659962,24.925325,64.208216,11.42337,0.0,41.542423,9.984809,0.00000,...,0.0,4.0,0.0,0.0,0.0,2.0,0.0,2.0,7.0,4.0,0.0,4.0,4.0,7.1212,121.5300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
1,1.0,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,388.684,340.300,388.381750,162.0,0.0,0.012887,-0.313407,0.313407,0.012887,0.607143,1.000000,1.285714,14.975110,9.689226,2.614066,-2.394690,2.658342,-2.444817,5.134527,0.120322,3.150503,1.080362,516.780124,20.208896,19.794682,19.794682,13.127794,12.595754,12.595754,12.648545,12.648545,11.473090,11.473090,8.180905,8.180905,-0.08,2.187

In [2]:
df['SI_log'] = np.log1p(df['SI'])

## Формирование таргета SI > медианы

In [3]:
median_si_log = df['SI_log'].median()
df['SI_gt_median'] = (df['SI_log'] > median_si_log).astype(int)

df['SI_log'].describe(), df['SI_gt_median'].value_counts(normalize=True)

(count    1001.000000
 mean        2.042131
 std         1.456218
 min         0.011424
 25%         0.889262
 50%         1.578185
 75%         2.866003
 max         9.656410
 Name: SI_log, dtype: float64,
 SI_gt_median
 0    0.5005
 1    0.4995
 Name: proportion, dtype: float64)

In [4]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI', 'SI_log', 'SI_gt_median'])
y = df['SI_gt_median']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [5]:
def clf_metrics(y_true, y_pred, y_proba=None):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
    return metrics

## Логистическая регрессия (база)

In [6]:
log_reg = LogisticRegression(max_iter=500, n_jobs=-1)
log_reg.fit(X_train_scaled, y_train)

y_val_pred_lr = log_reg.predict(X_val_scaled)
y_val_proba_lr = log_reg.predict_proba(X_val_scaled)[:, 1]
metrics_lr_val = clf_metrics(y_val, y_val_pred_lr, y_val_proba_lr)
metrics_lr_val

{'accuracy': 0.725,
 'f1_macro': 0.7249570245350836,
 'roc_auc': np.float64(0.7496875000000001)}

## RandomForestClassifier + RandomizedSearchCV

In [7]:
rf_clf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_clf_params = {
    'n_estimators': [200, 400, 600],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
}

rf_clf_search = RandomizedSearchCV(
    rf_clf,
    rf_clf_params,
    n_iter=25,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
rf_clf_search.fit(X_train, y_train)

rf_clf_search.best_params_, rf_clf_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'n_estimators': 200,
  'min_samples_split': 5,
  'min_samples_leaf': 1,
  'max_features': 0.5,
  'max_depth': 20},
 np.float64(0.7168822522194408))

In [8]:
rf_si_best = rf_clf_search.best_estimator_

y_val_pred_rf = rf_si_best.predict(X_val)
y_val_proba_rf = rf_si_best.predict_proba(X_val)[:, 1]
metrics_rf_val = clf_metrics(y_val, y_val_pred_rf, y_val_proba_rf)
metrics_rf_val

{'accuracy': 0.68125,
 'f1_macro': 0.6809384164222874,
 'roc_auc': np.float64(0.783515625)}

In [9]:
y_test_pred_rf = rf_si_best.predict(X_test)
y_test_proba_rf = rf_si_best.predict_proba(X_test)[:, 1]
metrics_rf_test = clf_metrics(y_test, y_test_pred_rf, y_test_proba_rf)
metrics_rf_test

{'accuracy': 0.6368159203980099,
 'f1_macro': 0.6324122554300173,
 'roc_auc': np.float64(0.6749999999999999)}

## XGBClassifier + RandomizedSearch

In [10]:
xgb_clf = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    n_estimators=400,
    n_jobs=-1,
)

xgb_clf_params = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

xgb_si_search = RandomizedSearchCV(
    xgb_clf,
    xgb_clf_params,
    n_iter=25,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
xgb_si_search.fit(X_train, y_train)

xgb_si_search.best_params_, xgb_si_search.best_score_

Fitting 3 folds for each of 25 candidates, totalling 75 fits


({'subsample': 0.8,
  'max_depth': 4,
  'learning_rate': 0.01,
  'colsample_bytree': 1.0},
 np.float64(0.7101606193394717))

In [11]:
xgb_si_best = xgb_si_search.best_estimator_

y_val_pred_xgb = xgb_si_best.predict(X_val)
y_val_proba_xgb = xgb_si_best.predict_proba(X_val)[:, 1]
metrics_xgb_val = clf_metrics(y_val, y_val_pred_xgb, y_val_proba_xgb)
metrics_xgb_val

{'accuracy': 0.69375,
 'f1_macro': 0.6931626942194044,
 'roc_auc': np.float64(0.77296875)}

In [12]:
y_test_pred_xgb = xgb_si_best.predict(X_test)
y_test_proba_xgb = xgb_si_best.predict_proba(X_test)[:, 1]
metrics_xgb_test = clf_metrics(y_test, y_test_pred_xgb, y_test_proba_xgb)
metrics_xgb_test

{'accuracy': 0.6318407960199005,
 'f1_macro': 0.6285214785214785,
 'roc_auc': np.float64(0.7033168316831683)}

## Вывод